# Semantic Continuity + NLI Completion Scoring

**Upload these files to `/content/sample_data/` before running:**

| File | Where to find it locally |
|---|---|
| `transcript_grp-*.tsv` (all transcript files) | `transcripts/final/` |
| `completion_candidates_broad.tsv` | `analysis/results/` |

**Outputs saved to `/content/` — download when done:**
- `semantic_continuity_window_30s.tsv`
- `nli_window_30s.tsv`
- `completion_scores_nli.tsv`

In [ ]:
!pip install -q sentence-transformers transformers sentencepiece accelerate

In [ ]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

TRANSCRIPT_DIR = Path('/content/sample_data')
OUT_DIR        = Path('/content')
WINDOW_S       = 30.0
PARTICIPANTS   = ('P1', 'P2', 'P3', 'P4')
_FNAME_RE      = re.compile(r'transcript_(grp[_-]?\d+)_(T\d+)_', re.IGNORECASE)

def norm_grp(raw):
    digits = re.sub(r'^grp[_-]?', '', raw.lower())
    return f'grp-{int(digits):02d}'

all_events = []
for fpath in TRANSCRIPT_DIR.glob('*.tsv'):
    m = _FNAME_RE.search(fpath.name)
    if not m:
        continue
    df = pd.read_csv(fpath, sep='\t')
    df['group_id'] = norm_grp(m.group(1))
    df['task_id']  = m.group(2).upper()
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
spk = events_df[
    (events_df['type'] == 'SPK') &
    (events_df['speaker'].isin(PARTICIPANTS)) &
    (events_df['text'].notna()) &
    (events_df['text'].str.strip() != '') &
    (events_df['task_id'] != 'T4')
].copy()
spk['text'] = spk['text'].str.strip()
spk['window_index'] = (spk['onset'] / WINDOW_S).apply(
    lambda x: int(x) if pd.notna(x) and x >= 0 else -1
)
spk = spk[spk['window_index'] >= 0].sort_values(['group_id', 'task_id', 'onset']).reset_index(drop=True)
print(f'Utterances: {len(spk)}')

## 1. Sentence-BERT Embeddings + Cosine Similarity per Window

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU " + torch.cuda.get_device_name(0) if device == 0 else "CPU"}')

sbert = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = sbert.encode(spk['text'].tolist(), show_progress_bar=True, batch_size=128, device='cuda' if device==0 else 'cpu')
spk['_emb_idx'] = range(len(spk))
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
win_rows = []
for (grp, task, win), win_df in spk.groupby(['group_id', 'task_id', 'window_index']):
    win_df = win_df.sort_values('onset').reset_index(drop=True)
    embs = embeddings[win_df['_emb_idx'].values]

    consecutive_sims, cross_sims = [], []
    speakers = win_df['speaker'].values
    for i in range(len(win_df) - 1):
        if speakers[i] != speakers[i+1]:
            consecutive_sims.append(float(cosine_similarity([embs[i]], [embs[i+1]])[0, 0]))
    for i in range(len(win_df)):
        for j in range(i+1, len(win_df)):
            if speakers[i] != speakers[j]:
                cross_sims.append(float(cosine_similarity([embs[i]], [embs[j]])[0, 0]))

    win_rows.append({
        'group_id': grp, 'task_id': task, 'window_index': win,
        'tr_semantic_continuity':      np.mean(consecutive_sims) if consecutive_sims else np.nan,
        'tr_cross_speaker_similarity': np.mean(cross_sims)       if cross_sims else np.nan,
    })

cont_df = pd.DataFrame(win_rows)
cont_df.to_csv(f'{OUT_DIR}/semantic_continuity_window_30s.tsv', sep='\t', index=False)
print(f'Saved {len(cont_df)} windows')
print(cont_df[['tr_semantic_continuity','tr_cross_speaker_similarity']].describe().round(3))

## 2. Zero-Shot NLI — Completion and Repetition Scores

In [ ]:
from transformers import pipeline

# Upload completion_candidates_broad.tsv to /content/sample_data/ before running this cell
cand_path = TRANSCRIPT_DIR / 'completion_candidates_broad.tsv'
if not cand_path.exists():
    raise FileNotFoundError(
        f"Upload 'completion_candidates_broad.tsv' from analysis/results/ "
        f"to /content/sample_data/ and re-run."
    )

cands = pd.read_csv(cand_path, sep='\t', index_col=0)
cands.columns = cands.columns.str.strip()
for col in ['text_a', 'text_b']:
    cands[col] = cands[col].astype(str).str.strip()
print(f'Candidates: {len(cands)}')

zsc = pipeline('zero-shot-classification',
               model='MoritzLaurer/deberta-v3-base-zeroshot-v1',
               device=device)
print('Classifier ready.')

In [ ]:
HYPOTHESES = [
    'the second speaker is completing the first speaker\'s unfinished sentence',
    'the second speaker is repeating or echoing what the first speaker said',
]

texts = [
    f'{r["speaker_a"]} said: "{r["text_a"]}" and then {r["speaker_b"]} said: "{r["text_b"]}"'
    for _, r in cands.iterrows()
]

BATCH = 64 if device == 0 else 16
results = []
for i in tqdm(range(0, len(texts), BATCH), desc='NLI scoring'):
    br = zsc(texts[i:i+BATCH], candidate_labels=HYPOTHESES, multi_label=True)
    if isinstance(br, dict):
        br = [br]
    results.extend(br)

for i, res in enumerate(results):
    scores = dict(zip(res['labels'], res['scores']))
    cands.at[cands.index[i], 'nli_completion_score'] = scores.get(HYPOTHESES[0], 0.0)
    cands.at[cands.index[i], 'nli_repetition_score'] = scores.get(HYPOTHESES[1], 0.0)

cands.to_csv(f'{OUT_DIR}/completion_scores_nli.tsv', sep='\t', index_label='#')
print(f'Saved NLI scores')
print(cands[['nli_completion_score','nli_repetition_score']].describe().round(3))

## 3. Aggregate NLI Scores to Window Level

In [ ]:
cands['window_index'] = (cands['onset_b'] / WINDOW_S).apply(
    lambda x: int(x) if pd.notna(x) and x >= 0 else -1
)

win_nli = cands[cands['window_index'] >= 0].groupby(
    ['group_id', 'task_id', 'window_index']
)[['nli_completion_score', 'nli_repetition_score']].mean().reset_index()
win_nli.columns = ['group_id', 'task_id', 'window_index',
                   'tr_nli_completion_score', 'tr_nli_repetition_score']

win_nli.to_csv(f'{OUT_DIR}/nli_window_30s.tsv', sep='\t', index=False)
print(f'Window-level NLI features: {len(win_nli)} rows')
print(win_nli[['tr_nli_completion_score','tr_nli_repetition_score']].describe().round(3))